# Anomaly detection (Isolation Forest)



_Classical ML_



**Outliers are easy to fence off. Few random cuts isolate them.**



An Isolation Forest finds outliers by trying to isolate each point with random cuts.



     Pick a feature at random, then a random split value. Repeat, narrowing down to one point.



---



This notebook builds an Isolation Forest one step at a time: first a toy blob with a few planted outliers, then real tumour scans. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.

import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Make a toy dataset with planted outliers

We generate a dense cloud of 200 "normal" points clustered tightly around the origin, then scatter 10 "outliers" uniformly across a much wider range. Stacking them gives one array `X` where the last 10 rows are the known anomalies — that ground truth lets us check the model afterwards.

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(0)                  # seeded for reproducibility

inliers = rng.normal(0, 1, size=(200, 2))       # dense blob centred at the origin
outliers = rng.uniform(-6, 6, size=(10, 2))     # 10 points scattered far out

# Stack them: rows 0-199 are normal, rows 200-209 are the planted outliers.
X = np.vstack([inliers, outliers])

### Step 2 — Fit the Isolation Forest and score every point

`IsolationForest` builds many random trees that repeatedly split on a random feature at a random threshold. Outliers get isolated in just a few cuts, so they sit at shallow tree depth. `predict` labels each point `-1` (anomaly) or `1` (normal); `decision_function` gives a continuous score where **lower means more anomalous**. `contamination=0.05` tells it to expect about 5% anomalies.

In [ ]:
iso = IsolationForest(contamination=0.05, random_state=0).fit(X)

pred = iso.predict(X)                # -1 = anomaly, 1 = normal
score = iso.decision_function(X)     # lower = more anomalous

### Step 3 — Read off what got flagged

We find the indices the forest flagged, then count how many of the 10 true outliers (indices 200 and up) it actually caught. The single most anomalous point is the one with the lowest decision-function score.

In [ ]:
flagged = np.where(pred == -1)[0]
caught = sum(i >= 200 for i in flagged)     # true outliers live at index >= 200
most_anomalous = int(np.argmin(score))      # lowest score = most anomalous

print("points flagged as anomalies:", len(flagged))
print("of the 10 true outliers, caught:", caught)
print("most anomalous index:", most_anomalous,
      " score=%.3f" % score.min())

## Visualize the data & results

_On real breast-cancer scans, which tumors look like outliers?_

### Step 1 — Fit the forest on real tumour scans

Now we swap the toy blob for real data: two columns of the breast-cancer dataset (mean radius and mean texture). We fit a fresh Isolation Forest on these two features, raising `contamination` to 0.08 since real measurements have more genuine spread.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import IsolationForest

bc = load_breast_cancer()
X = bc.data[:, [0, 1]]                 # mean radius, mean texture (real features)

iso = IsolationForest(contamination=0.08, random_state=0).fit(X)

### Step 2 — Split points into normal vs flagged

We run `predict` and use the `-1` / `1` labels to slice the data into two groups, so the plot can colour them differently.

In [ ]:
pred = iso.predict(X)                  # -1 = anomaly, 1 = normal

normal = X[pred == 1]
flagged = X[pred == -1]

### Step 3 — Plot the scans in feature space

Blue points are the bulk of scans the forest considers normal; red points are the ones it flagged as outliers. They sit at the edges of the cloud — the unusually large or oddly textured tumours.

In [ ]:
plt.scatter(normal[:, 0], normal[:, 1], c="#4ea1ff", s=12, label="normal")
plt.scatter(flagged[:, 0], flagged[:, 1], c="#ff7b72", s=12, label="anomaly (flagged)")
plt.title("Isolation Forest on breast cancer: normal scans vs flagged outliers")
plt.xlabel("mean radius")
plt.ylabel("mean texture")
plt.legend()
plt.show()